# Flag-Complex ECP

This example shows how to compute the Euler Characteristic Profile (ECP) of the flag complex (aka clique complex) of a filtered graph using `pyEulerCurves`. For one-dimensional filtration, this returns the usual Euler Characteristic Curve (ECC) of the flag complex.

The input is a graph with filtration values on the vertices and edges, either as a `pyEulerCurves.FilteredGraph` or a `networkx.Graph` with filtration annotations.

### pyEulerCurves.FilteredGraph

In [ ]:
from pyEulerCurves import ECP_from_filtered_graph, FilteredGraph

graph = FilteredGraph(
    vertex_filtrations=[(0.0, 0.0), (0.0, 0.0), (0.0, 0.0)],
    edges=[(0, 1), (0, 2), (1, 2)],
    edge_filtrations=[(0.25, 0.5), (0.5, 0.25), (0.75, 0.75)],
)

transformer = ECP_from_filtered_graph()
ecp = transformer.fit_transform(graph)

print(ecp)

# Simplex statistics are available as attributes after transform.
print("num_simplices:", transformer.num_simplices)
print("largest_dimension:", transformer.largest_dimension)
print("dim_counts:", transformer.dim_counts)

### networkx.Graph

In [ ]:
import networkx as nx
from pyEulerCurves import ECP_from_filtered_graph

graph = nx.Graph()
graph.add_node("a", filtration=(0.0, 0.0))
graph.add_node("b", filtration=(0.0, 0.0))
graph.add_node("c", filtration=(0.0, 0.0))
graph.add_edge("a", "b", filtration=(0.25, 0.5))
graph.add_edge("a", "c", filtration=(0.5, 0.25))
graph.add_edge("b", "c", filtration=(0.75, 0.75))

ecp = ECP_from_filtered_graph().fit_transform(graph)
print(ecp)

## Parallel Computation

The ECP computation can be parallelized by splitting it across the graph's vertices.

For this, pass `workers` to the `ECP_from_filtered_graph` transformer: `workers=1` (the default) runs sequentially, `workers=-1` uses all available logical CPUs, and any positive integer sets an explicit worker count.

In [ ]:
# For a large graph, use all available cores:
ecp = ECP_from_filtered_graph(workers=-1).fit_transform(graph)
print(ecp)

### When to use parallelization
In our tests on a standard laptop (4 CPUs, 16 GB RAM) parallel becomes the better choice once the sequential ECP takes more than roughly **one second**; below that, the sequential version is faster because of the process-pool startup cost.

The figures below show a sweep over random Erdős–Rényi filtered graphs over a range of sizes (`n`) and edge densities (`p`) and compare `workers=-1` against `workers=1`. The speedup is the ratio of sequential to parallel runtime, so values above 1 indicate a win for parallelization. In the left (decision) figure, every point is one example graph, and the orange curve is a logistic regression for the probability that parallel is faster as a function of the sequential runtime. The grey line marks the estimated **50% break-even** point and the shaded band is its 95% confidence interval. The point estimate lies a bit *below* a second; the ~1 second rule of thumb is near the upper end of the 95% confidence interval. In other words, we suggest to switch to parallel once the sequential runtime clears the upper confidence bound of the break-even, so that we are confident the job is around break-even.

We suggest to run your own sweep on your own hardware to find the break-even point for your use case, for example using the `bench_ecp_flag.py` script in the `examples` directory (~5 minutes).

<img src="data/ecp_flag_decision.png" height="300" alt="Decision plot.">
<img src="data/ecp_flag_sweep.png" height="300" alt="Speedup over number of vertices and edge density.">

## Coarsened Filtrations

For large graphs with a lot of different floating point filtration values, the number of distinct contributions to the ECP can grow very large. In this case, it may be desirable to coarsen the filtration values before computing the ECP, for example by rounding every filtration value to a common grid. This functionality is not provided by `pyEulerCurves`, as it is best to apply this preprocessing step to the input graph before passing it to `ECP_from_filtered_graph`.

The example below builds a random graph with continuous filtration values. It then compares the ECP computed with the original filtration to the ECP computed after rounding every value to 2 decimal. The two curves are very close, but the coarsened version requires less memory and is often faster to compute because it has fewer distinct contributions to iterate over during the ECP collection step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyEulerCurves import ECP_from_filtered_graph, FilteredGraph

def ecc_curve(ecp):
    """Cumulative Euler characteristic vs. (1-D) filtration value."""
    values = [filtration[0] for filtration, _ in ecp]
    euler = np.cumsum([contribution for _, contribution in ecp])
    return values, euler



# A random graph with continuous (high-resolution) filtration values.
rng = np.random.default_rng(0)
n = 300
iu, ju = np.triu_indices(n, k=1)
keep = rng.random(iu.size) < 0.05
sources, targets = iu[keep], ju[keep]
edges = list(zip(sources.tolist(), targets.tolist()))

vertex_filt = rng.random(n)
# Edge filtration >= max of its endpoints, so the filtration is valid.
edge_filt = np.maximum(vertex_filt[sources], vertex_filt[targets]) + rng.random(len(edges))

fine = FilteredGraph(vertex_filt.tolist(), edges, edge_filt.tolist())
fine_ecp = ECP_from_filtered_graph()
ecp_fine = fine_ecp.fit_transform(fine)
print(f"{fine_ecp.num_simplices} simplices in the flag complex")
print(f"distinct ECP values (fine):   {len(ecp_fine)}")

x_fine, y_fine = ecc_curve(ecp_fine)

# Coarsen: round every filtration value to two decimals.
decimals = 2
coarse = FilteredGraph(
    np.round(vertex_filt, decimals).tolist(),
    edges,
    np.round(edge_filt, decimals).tolist(),
)
ecp_coarse = ECP_from_filtered_graph().fit_transform(coarse)
print(f"distinct ECP values (coarse): {len(ecp_coarse)}")

x_coarse, y_coarse = ecc_curve(ecp_coarse)

# Plot the original and coarsened ECCs.
fig, ax = plt.subplots(figsize=(7, 4))
ax.step(x_fine, y_fine, where="post", lw=2, label=f"fine ({len(ecp_fine)} values)")
ax.step(x_coarse, y_coarse, where="post", lw=2, label=f"coarse ({len(ecp_coarse)} values)")
ax.set_xlabel("filtration value")
ax.set_ylabel("Euler characteristic")
ax.set_title("ECC: original vs. coarsened filtration")
ax.legend()
plt.show()